In [ ]:
%matplotlib inline
import skimage.io
import skimage.measure
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import os, pickle
from glob import glob
import scipy.ndimage
import shutil
import random
from skimage.measure import label, regionprops
from skimage.morphology import label, binary_closing
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import json, pylab, math
from scipy.optimize import curve_fit
from scipy.stats import norm
from shutil import copy2
from fastai.vision.all import *
from fastai.callback.hook import *
from fastai.test_utils import *

In [ ]:
#Define groupings
number_of_object_classes = 3
length_of_object_classes = [5,3,1]
labels = ['nucleolus','outoffocus','nosignal']
if len(labels) is not number_of_object_classes:
    print('Not the correct number of labels!')

In [ ]:
#Function to process annotations
def process_annotation(path, number_of_object_classes, length_of_object_classes, labels):
    
    #import annotation
    annotation = skimage.io.imread(path)
    if len(annotation.shape) == 3:
        annotation = annotation[:,:,0]
    
    #set groupings
    cumsum = 1
    groupings = [[] for i in range(number_of_object_classes)]
    for i in range(number_of_object_classes):
        groupings[i] = list(range(cumsum,cumsum+length_of_object_classes[i]))
        cumsum = cumsum + length_of_object_classes[i]
    
    #produce masks for each object class
    masks = {}
    
    for i in range(number_of_object_classes):
        grouping = groupings[i]
        masks[labels[i]] = []
        for value in grouping:
            differentiated_masks = skimage.measure.label(annotation == value)
            n_masks = np.max(differentiated_masks)
            for j in range(1,n_masks+1):
                mask = np.asarray((differentiated_masks==j),dtype='uint8')
                mask = scipy.ndimage.morphology.binary_closing(mask)
                mask = 255*scipy.ndimage.morphology.binary_fill_holes(mask)
                masks[labels[i]].append(mask)
                
    return masks

# Train

In [ ]:
outdir = Path('/home/hyeonjin/projects/annotation_processing/hjk_nucleolin_40xdry_v1/nucleolin_filter_classifier/images_train_3_classes/')

data = ImageDataLoaders.from_folder(path = outdir, #vocab = ['dots', 'edge', 'noise', 'signal'],
                                    train = 'train', valid = 'val',
                                    item_tfms=Resize((224,224)),
                                    batch_tfms=[*aug_transforms(do_flip=False), Normalize.from_stats(*imagenet_stats)])
data


In [ ]:
data.show_batch(figsize=(7,6))

In [ ]:
learn_34 = cnn_learner(data, models.resnet34, metrics=accuracy)

In [ ]:
learn_34.lr_find()

In [ ]:
learn_34.fit_one_cycle(10, 1e-3)

In [ ]:
interp = ClassificationInterpretation.from_learner(learn_34)

In [ ]:
interp.plot_top_losses(36, figsize=(24,18))

In [ ]:
interp.plot_confusion_matrix(figsize=(7,7),dpi=90)

In [ ]:
learn_34.unfreeze()

In [ ]:
learn_34.fit_one_cycle(30, 1e-5)

In [ ]:
learn_34.fit_one_cycle(30, 1e-5)

In [ ]:
interp = ClassificationInterpretation.from_learner(learn_34)

In [ ]:
learn_34.export('export.pkl')

## Classifier performance

In [ ]:
outdir = Path('/home/hyeonjin/projects/annotation_processing/hjk_nucleolin_40xdry_v1/nucleolin_filter_classifier/images_train_3_classes/')

data = ImageDataLoaders.from_folder(
    path = outdir, 
    train = 'train', 
    valid = 'val',
    item_tfms=Resize((224,224)),
    batch_tfms=[*aug_transforms(do_flip=False), Normalize.from_stats(*imagenet_stats)]
)

classification_model_path = '/home/hyeonjin/trapnell-learning/Projects/fastai_models/nucleolin_filter_classifier/images_train_3_classes/export.pkl'
learn = load_learner(classification_model_path, cpu=False)
learn.dls = data

# 3. Verify it worked (should no longer be 0)
print(f"Validation images found: {len(learn.dls.valid_ds)}")

# 4. Now the standard interpretation will work!
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix(figsize=(7,7), dpi=150)

In [ ]:
## Supplementary Figure 5A

preds, targs = learn.get_preds(dl=learn.dls.valid)
decoded = preds.argmax(dim=-1)

new_labels = ['Boundary', 'No signal', 'Signal']

cm = confusion_matrix(targs, decoded)
fig, ax = plt.subplots(figsize=(7, 7), dpi=300)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=new_labels)
disp.plot(cmap='Blues', ax=ax, values_format='d')

plt.title("Nuclear compartment signal classifier", fontsize=14)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('Actual Label', fontsize=12)
plt.xticks(rotation=45)
plt.show()

In [ ]:
## Supplementary Figure 5D

preds, targs = learn.get_preds(dl=learn.dls.valid)
decoded = preds.argmax(dim=-1)

new_labels = ['Boundary', 'No signal', 'Signal']

correct_idxs = (decoded == targs).nonzero(as_tuple=True)[0]

fig, axes = plt.subplots(3, 3, figsize=(12, 12), dpi=150)

for class_idx in range(3):
    class_correct = [i.item() for i in correct_idxs if targs[i] == class_idx]
    num_to_display = min(len(class_correct), 3)
    random_sample = random.sample(class_correct, num_to_display)
    
    for i in range(3):
        ax = axes[class_idx, i]
        if i < len(random_sample):
            idx = random_sample[i]
            img, _ = learn.dls.valid_ds[idx]
            img.show(ctx=ax)
            ax.set_title("")
        else:
            ax.text(0.5, 0.5, 'No more examples', ha='center', va='center')
            
        ax.axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()